# Проверка исправности данных до анализа

<b>Задача данного ноутбука - проверить пропуски, дубликаты и аномалии в сырых данных</b>

## setup

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import duckdb
from src.functions import fancy_info, detect_anomaly

#### импорт данных

In [3]:
RAW = PROJECT_ROOT / 'data' / 'raw'

details_df = pd.read_csv(RAW / 'details.csv', sep=',')
stats_df = pd.read_csv(RAW / 'stats.csv', sep=',')

In [4]:
# ratings присоединим через duckdb, т.к. она слишком большая
con = duckdb.connect()
con.execute("SET enable_progress_bar=false")

path = (RAW / 'ratings.csv').as_posix()
parquet_path = (RAW / 'ratings.parquet').as_posix()

# посмотреть схему
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

# конвертируем в parquet для ускорения последующих запросов
if not (RAW / 'ratings.parquet').exists():
    con.sql(f"COPY (SELECT * FROM read_csv_auto('{path}')) TO '{parquet_path}' (FORMAT PARQUET)")

# создаём view на этом файле
con.sql(f"CREATE VIEW ratings AS SELECT * FROM read_parquet('{parquet_path}')")

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ username             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ anime_id             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ status               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ score                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_rewatching        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ num_watched_episodes │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



## Обзор структуры таблиц (нули, пропуски, типы данных)

### details

In [5]:
# details
fancy_info(details_df)

,column,dtype,non_null_count,null_percent
0,mal_id,int64,28955,0.00
1,title,object,28955,0.00
2,title_japanese,object,28836,0.41
3,url,object,28955,0.00
4,image_url,object,28955,0.00
5,type,object,28888,0.23
6,status,object,28955,0.00
7,score,float64,18882,34.79
8,scored_by,float64,18882,34.79
9,start_date,object,28104,2.94


Для начала уберём поля, которые точно не понадобятся. Для details это `title_japanese` - название на японском, `url` - ссылка, `image_url` - ссылка на изображения, `producers`, `licensors`, `streaming` - относятся к японскому рынку, `explicit_genres` - все нули

In [6]:
details = details_df.copy()
stats = stats_df.copy()
details = details.drop(['title_japanese', 'url', 'image_url', 'producers', 'licensors', 'streaming', 'explicit_genres'], axis=1)

Посмотрим нули входных признаков аниме. `type` - 0.2% отлично, `genre` - 21% нормально,`themes` - 41% приемлемо, `demographics` - 62% слишком разреженный, лучше не брать, `source` - нет нулей, отлично, `rating` 2% отлично, `episodes` 2% отлично, `year` - 78% очень странно, зато в `start_date` всего 3% нулей. Так что возьмём start_date. Type, genre, themes, source, rating нужно поменять на str, start_date - на datetime

In [7]:
details['type'] = details['type'].astype('string')
details['genres'] = details['genres'].astype('string')
details['themes'] = details['themes'].astype('string')
details['source'] = details['source'].astype('string')
details['rating'] = details['rating'].astype('string')

# перед сменой типа start_date на дату, проверим, что все непустые значения записаны в одном формате: 2010-01-01T00:00:00+00:00
pattern = r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\+00:00$'
filled = details['start_date'].notna()
bad_format = details.loc[filled & ~details['start_date'].str.match(pattern, na=False), 'start_date']

print(f'непустых значений: {filled.sum()}')
print(f'не подходят под формат: {len(bad_format)}')

непустых значений: 28104
не подходят под формат: 0


In [8]:
details['start_date'] = pd.to_datetime(details['start_date'], errors='coerce')

print('тип:', details['start_date'].dtype)
print('пропусков:', details['start_date'].isna().sum())

тип: datetime64[ns, UTC]
пропусков: 851


`score` в details - 35% пропусков. Шкала MAL - от 1 до 10, так что ноль на ней в принципе невозможен, но проверим по данным.

In [9]:
print(f"пропусков в score: {details['score'].isna().sum()} ({details['score'].isna().mean()*100:.2f}%)")
print(f"значений ровно 0:  {(details['score'] == 0).sum()}")
print(f"диапазон score:    от {details['score'].min()} до {details['score'].max()}")

# если оценки нет, то и число оценивших должно быть пустым
no_score = details['score'].isna()
print(f"из них scored_by тоже пустой: {details.loc[no_score, 'scored_by'].isna().sum()} из {no_score.sum()}")

пропусков в score: 10073 (34.79%)
значений ровно 0:  0
диапазон score:    от 1.89 до 9.29
из них scored_by тоже пустой: 10073 из 10073


Подтвердилось: значений ровно 0 нет ни одного, минимум 1.89, и у всех 10073 тайтлов без `score` поле `scored_by` тоже пустое. Значит пропуск = оценки нет а не оценка 0. Такие тайтлы из анализа удовлетворённости просто исключим

Но отсюда не следует, что эти аниме никто не оценивал. Сверим с распределением голосов в stats.

In [10]:
votes = [f'score_{i}_votes' for i in range(1, 11)]
score_check = pd.DataFrame({
    'mal_id': stats['mal_id'],
    'голоса в stats': np.where(stats[votes].isna().all(axis=1), 'пусто', 'есть'),
}).merge(details[['mal_id', 'score']], on='mal_id')
score_check['score в details'] = np.where(score_check['score'].isna(), 'пусто', 'есть')

pd.crosstab(score_check['голоса в stats'], score_check['score в details'])

score в details,есть,пусто
голоса в stats,,
есть,18882,9643
пусто,0,430


Правый нижний угол пуст - если голосов нет, то и `score` всегда пустой, тут всё логично. А вот 9643 тайтла имеют в stats непустое распределение голосов, но `score` в details у них отсутствует. То есть люди голосовали, а агрегированной оценки нет.

Раз голоса есть - может, восстановить `score` как средневзвешенное? Посмотрим на эти тайтлы поближе.

In [11]:
# считаем средневзвешенную оценку из распределения голосов: sum(оценка * голоса) / sum(голоса)
sum_votes = stats[votes].sum(axis=1)
mean_votes = stats[votes].mul(np.arange(1, 11), axis=1).sum(axis=1) / sum_votes.replace(0, np.nan)

check = details[['mal_id', 'title', 'members', 'score', 'status']].merge(
    pd.DataFrame({'mal_id': stats['mal_id'], 'голосов': sum_votes, 'score_из_голосов': mean_votes.round(2)}),
    on='mal_id')

# самые популярные из тех, где score пустой, а голоса есть
check[check['score'].isna() & check['голосов'].gt(0)].nlargest(5, 'members')

,mal_id,title,members,score,status,голосов,score_из_голосов
17994,52807,One Punch Man 3,327719,NaN,Not yet aired,11.0,8.45
23117,59978,Sousou no Frieren 2nd Season,199585,NaN,Not yet aired,18.0,9.17
23219,59027,Spy x Family Season 3,180909,NaN,Not yet aired,74.0,8.49
10670,55825,Jigokuraku 2nd Season,175937,NaN,Not yet aired,4.0,9.00
16470,59193,Mushoku Tensei III: Isekai Ittara Honki Dasu,138674,NaN,Not yet aired,2.0,9.00


Картина проясняется: например у `One Punch Man 3` 327 тысяч человек в списках, а голосов всего 11. Это ожидаемые продолжения, которые ещё не вышли - люди добавляют их в план, а оценки ставят единицы (видимо, по трейлеру). MAL такую оценку не публикует, 11 голосов на 327 тысяч ожидающих ничего не говорят о качестве. Лучше ничего не делать с `score`, так как мы не будем оценивать невышедшие аниме. Их популярность, если она есть, обусловлена популярностью предыдущих сезонов, а у нас будет новое аниме - совсем другая ситуация.

### stats

In [12]:
# stats
fancy_info(stats_df)

,column,dtype,non_null_count,null_percent
0,mal_id,int64,28955,0.00
1,watching,int64,28955,0.00
2,completed,int64,28955,0.00
3,on_hold,int64,28955,0.00
4,dropped,int64,28955,0.00
5,plan_to_watch,int64,28955,0.00
6,total,int64,28955,0.00
7,score_1_votes,float64,28525,1.49
8,score_1_percentage,float64,28525,1.49
9,score_2_votes,float64,28525,1.49


В stats ничего удалять не будем - все поля теоретически могут понадобиться для метрик охвата, удовлетворённости и удержания. Есть пропуски в блоке оценок (1.49%)  `score_x_votes`, `score_x_percentage`, и они одинаковые во всех 20 колонках. Скорее всего это одни и те же строки. Проверим

In [13]:
stats = stats_df.copy()
votes = [f'score_{i}_votes' for i in range(1, 11)]
pcts = [f'score_{i}_percentage' for i in range(1, 11)]

# сколько из 20 колонок с оценками пустует в каждой строке
na_per_row = stats[votes + pcts].isna().sum(axis=1) # складываем т к 0 голосов даёт 0%
na_per_row.value_counts().rename_axis('пустых колонок из 20').to_frame('строк') 

,строк
пустых колонок из 20,
0,28525
20,430


Промежуточных случаев нет: 28525 строк заполнены полностью, 430 пустые целиком. Значит это тайтлы, по которым нет ни одного голоса. Посмотрим, что это за тайтлы.

In [14]:
no_votes = stats[votes].isna().all(axis=1) # id тайтлов с нулями в оценках
no_score = details['mal_id'].isin(stats.loc[no_votes, 'mal_id']) # ищем их в details

# посмотрим их статус
print(details.loc[no_score, 'status'].value_counts().to_frame('тайтлов'))

# сравним их с остальными по количеству добавлений в списки (details.members)
pd.DataFrame({
    'без оценок': [no_score.sum(), details.loc[no_score, 'members'].median()],
    'с оценками': [(~no_score).sum(), details.loc[~no_score, 'members'].median()],
}, index=['тайтлов', 'медиана members'])

               тайтлов
status                
Not yet aired      430


,без оценок,с оценками
тайтлов,430.0,28525.0
медиана members,1972.0,1060.0


Все 430 - `Not yet aired`, то есть ещё не вышедшие. Оценивать пока нечего, при этом в списки их активно добавляют, видимо в plan_to_watch. Так как тут нет никакой ошибки, не будем убирать их из анализа

Типы данных в stats корректные

### ratings

In [15]:
# ratings
print("ratings")
con.sql("DESCRIBE SELECT * FROM ratings").show()
con.sql("SELECT count(*) AS row_count FROM ratings").show()

ratings
┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ username             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ anime_id             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ status               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ score                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_rewatching        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ num_watched_episodes │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│ 124298357 │
└───────────┘



Во-первых, посмотрим нули, во-вторых, удалять здесь нечего - все шесть полей нужны. Сразу бросается в глаза поле `is_rewatching` - важный показатель лояльности, уникальных для этой таблицы

In [16]:
con.sql("""
    SELECT 'username' AS поле, count(*)-count(username) AS пропусков, count(DISTINCT username) AS уникальных FROM ratings
    UNION ALL SELECT 'anime_id', count(*)-count(anime_id), count(DISTINCT anime_id) FROM ratings
    UNION ALL SELECT 'status', count(*)-count(status), count(DISTINCT status) FROM ratings
    UNION ALL SELECT 'score', count(*)-count(score), count(DISTINCT score) FROM ratings
    UNION ALL SELECT 'is_rewatching', count(*)-count(is_rewatching), count(DISTINCT is_rewatching) FROM ratings
    UNION ALL SELECT 'num_watched_episodes', count(*)-count(num_watched_episodes), count(DISTINCT num_watched_episodes) FROM ratings
""").show()

┌──────────────────────┬───────────┬────────────┐
│         поле         │ пропусков │ уникальных │
│       varchar        │   int64   │   int64    │
├──────────────────────┼───────────┼────────────┤
│ username             │         7 │     309313 │
│ anime_id             │         0 │      29271 │
│ status               │         0 │          6 │
│ score                │         0 │         11 │
│ is_rewatching        │   3797321 │          2 │
│ num_watched_episodes │         0 │       1919 │
└──────────────────────┴───────────┴────────────┘



Пропусков почти нет: 7 строк без `username`, это мелочь. 3.8 млн без `is_rewatching` - это ~3% из 130M, вернёмся к ним позже

Странно, что в ratings **29271 уникальных аниме** по `anime_id`, хотя в details всего 28955 тайтлов. То есть в оценках встречаются аниме, для которых нет карточки. Посмотрим, как такое вышло

In [17]:
# отдадим duckdb список id из details, чтобы соединить с ratings
con.register('details_ids', details[['mal_id']])

con.sql("""
    SELECT count(DISTINCT r.anime_id) AS аниме_без_карточки,
           count(*) AS строк,
           round(100.0 * count(*) / (SELECT count(*) FROM ratings), 2) AS доля_строк
    FROM ratings r
    LEFT JOIN details_ids d ON r.anime_id = d.mal_id
    WHERE d.mal_id IS NULL
""").show()

┌────────────────────┬────────┬────────────┐
│ аниме_без_карточки │ строк  │ доля_строк │
│       int64        │ int64  │   double   │
├────────────────────┼────────┼────────────┤
│                326 │ 307757 │       0.25 │
└────────────────────┴────────┴────────────┘



326 аниме без карточки, на них приходится 307757 строк - 0.25% всех оценок. Вероятно, тайтлы удалили или перенесли на MAL уже после того, как собрали датасет. При inner join с details эти строки отвалятся, потеря в четверть процента приемлема.

Теперь проверим, есть ли в столбцу `score` нули. По-идее, аниме на MyAnimeList нельзя оценить на ноль, то есть 0 в оценке обозначает её отсутствие. В details пропуск оценки обозначался как NaN, а здесь весь столбец целочисленный и содержит 11 уникальных значений, видимо от 0 до 10. Нужно заменить эти нули на NULL, либо везде писать условие `WHERE score > 0` чтобы не искажать статистику

In [18]:
# посмотрим долю оценок 0, в разных статусах
con.sql("""
    SELECT status,
           count(*) AS строк,
           sum(CASE WHEN score = 0 THEN 1 ELSE 0 END) AS score_ноль,
           round(100.0 * sum(CASE WHEN score = 0 THEN 1 ELSE 0 END) / count(*), 1) AS процент_ноль
    FROM ratings
    GROUP BY status
    ORDER BY строк DESC
""").show()

┌───────────────┬──────────┬────────────┬──────────────┐
│    status     │  строк   │ score_ноль │ процент_ноль │
│    varchar    │  int64   │   int128   │    double    │
├───────────────┼──────────┼────────────┼──────────────┤
│ completed     │ 79138385 │   13629637 │         17.2 │
│ plan_to_watch │ 31631312 │   31418876 │         99.3 │
│ watching      │  5678167 │    4481721 │         78.9 │
│ dropped       │  4580651 │    2623841 │         57.3 │
│ on_hold       │  3253152 │    2522161 │         77.5 │
│ unknown       │    16690 │      16000 │         95.9 │
└───────────────┴──────────┴────────────┴──────────────┘



У досмотренных (completed) 17.2% нулей, это довольно хороший показатель. 83% пользователей, досмотревших аниме, оценивают его. 
В plan_to_watch 99% нулей - это логично. В dropped - 57% - меньше, чем у watching и on_hold. Стоит отметить, что status - это то, что пользователь отмечает сам, то есть в dropped попадают тайтлы, которые пользователь целенаправленно не хочет досматривать, закономерно, что он сразу может выставить оценку.

In [19]:
# сделаем view ratings с заменой численных 0 на NULL
con.sql("""
CREATE VIEW ratings_clean AS
SELECT username, anime_id, NULLIF(score, 0) AS score
FROM ratings;
""")

Осталось поближе рассмотреть нули в булевом is_rewatching

In [20]:
con.sql('SELECT is_rewatching, count(*) AS строк FROM ratings GROUP BY is_rewatching ORDER BY строк DESC').show()

┌───────────────┬───────────┐
│ is_rewatching │   строк   │
│    double     │   int64   │
├───────────────┼───────────┤
│           0.0 │ 120397695 │
│          NULL │   3797321 │
│           1.0 │    103341 │
└───────────────┴───────────┘



Флаг `is_rewatching` содержит 0.0 у 120.4 млн строк, 1.0 всего у 103 тысяч (0.08%), плюс 3.8 млн `NULL`. Я отмечала этот признак интересным, но не понимаю, откуда берётся эта информация, ведь MyAnimeList - это фан соцсеть, а не стримминговая платформа, то есть они не могут отследить этот показатель в полной мере. И у них нет возможности самому пользователю отметить аниме как пересматриваемое. Можно предположить, что 0 - значит смотрел, но не пересматривал, а NULL - не смотрел вообще. в ratings записаны взаимодействия пользователя с аниме, если они есть. То есть такой вариант реально возможен для аниме в списке plan_to_watch. Проверим это

In [21]:
con.sql("""
    SELECT status,
           count(*) AS 'NULL в is_rewatching'
    FROM ratings
    WHERE is_rewatching ISNULL
    GROUP BY status;
""")

┌───────────────┬──────────────────────┐
│    status     │ NULL в is_rewatching │
│    varchar    │        int64         │
├───────────────┼──────────────────────┤
│ dropped       │               210544 │
│ plan_to_watch │               317332 │
│ on_hold       │               170179 │
│ watching      │                75312 │
│ completed     │              3023954 │
└───────────────┴──────────────────────┘

Предположение не подтвердилось, больше всего null наоборот среди досмотренных аниме. Так что оставим в покое этот столбец

## Аномалии

### Details
Прежде чем вручную смотреть отдельные поля, прогоним `detect_anomaly`, специально написанную под этот датасет, по числовым столбцам pandas-таблиц. Функция отделяет жёсткие аномалии (значения на битовых пределах 2**n−1, заглушки, отрицательные счётчики) от справочных выбросов по IQR, чтобы длинный хвост не выдавался за ошибку.

In [18]:
# details
detect_anomaly(details)

,column,role,dtype,null_%,min,max,n_unique,tail,severity,issues,examples
0,mal_id,id,int64,0.00,1.000000,62590.000000,28955,1.000000,ok,ok,
1,score,score,float64,34.79,1.890000,9.290000,561,1.100000,ok,ok,
2,scored_by,count,float64,34.79,101.000000,2979733.000000,9245,5.500000,ok,ok,
3,rank,rank,float64,24.03,1.000000,22020.000000,17765,1.000000,hard,not_unique:4232,
4,popularity,rank,int64,0.00,1.000000,28999.000000,21954,1.000000,hard,not_unique:7001,
5,members,count,int64,0.00,23.000000,4230312.000000,12276,5.600000,ok,ok,
6,favorites,count,int64,0.00,0.000000,243358.000000,1984,29.000000,ok,ok,
7,episodes,count,float64,2.35,1.000000,3000.000000,261,23.000000,soft,spike:2,"200x19 (r=19.0), 365x19 (r=19.0)"
8,year,year,float64,78.36,1961.000000,2026.000000,66,1.000000,ok,ok,


Для поля rank наличие неуникальных значений значит, что тайтлам с одинаковой средней оценкой присваивается одинаковый ранг. 

In [19]:
rank_count = details.groupby('rank')['mal_id'].count()
dup_ranks = rank_count[rank_count > 1].index # значения rank с дублями

same_ranks = details[details['rank'].isin(dup_ranks)]
same_ranks.sort_values('rank')[['rank', 'title', 'status', 'score', 'scored_by']].head(20)

,rank,title,status,score,scored_by
14593,153.0,Mahou Shoujo Madoka★Magica Movie 3: Hangyaku n...,Finished Airing,8.50,242444.0
10874,153.0,JoJo no Kimyou na Bouken Part 4: Diamond wa Ku...,Finished Airing,8.50,842074.0
19827,171.0,Rainbow: Nisha Rokubou no Shichinin,Finished Airing,8.46,164609.0
28438,171.0,Yuu☆Yuu☆Hakusho,Finished Airing,8.46,350816.0
19047,178.0,Pluto,Finished Airing,8.45,100446.0
18520,178.0,Owarimonogatari,Finished Airing,8.45,247151.0
3344,180.0,Chainsaw Man,Finished Airing,8.44,1031406.0
23214,180.0,Spy x Family,Finished Airing,8.44,1102931.0
5451,182.0,Doupo Cangqiong: Nian Fan,Currently Airing,8.44,3821.0
7683,182.0,Grand Blue,Finished Airing,8.44,463699.0


Действительно, тайтлы с одинаковой средней оценкой имеют одинаковый ранг. Но если бы я это решала, я бы в случае равенства score давала ранг выше тайтлу с большим scored_by

Рассмотрим поближе неуникальные значения в popularity. Скорее всего, popularity зависит от members. Проверим это

In [20]:
dup_pop = details[details['popularity'].duplicated(keep=False)]
dup_pop.groupby('popularity')['members'].nunique().value_counts()

members
2    3163
1    2655
3     250
4       7
Name: count, dtype: int64

Гипотеза не подтвердилась — 56% дублей (3420 из 6075 групп) не объясняются равенством members. Стоит посмотреть, насколько сильно расходится members внутри таких групп — если расхождение мизерное, это можно списать на округление/пересчёт снапшота.

In [22]:
gap = dup_pop.groupby('popularity')['members'].agg(['min','max'])
gap = gap[gap['min'] != gap['max']]
gap['diff'] = gap['max'] - gap['min']
gap['diff_pct'] = gap['diff'] / gap['min'] * 100
gap.sort_values('diff', ascending=False).head(20)

,min,max,diff,diff_pct
popularity,,,,
918,292799,293713,914,0.312160
812,327301,327954,653,0.199511
727,362355,362884,529,0.145989
640,405682,406111,429,0.105748
1940,127991,128333,342,0.267206
964,278686,279000,314,0.112672
337,687814,688037,223,0.032422
1339,199385,199585,200,0.100308
2329,98884,99078,194,0.196189


Процент гэпа везде маленький, похоже на рассинхрон сбора данных, но это всё ещё не подтверждает связи popularity с members. Вообще, popularity - это внутренний критерий MAL, содержание которого они не раскрывают. Чтобы проверить связь строго, можно посчитать ранговую корреляцию по всей таблице

In [23]:
from scipy.stats import spearmanr
spearmanr(details['members'], details['popularity'])

SignificanceResult(statistic=np.float64(-0.999999145861841), pvalue=np.float64(0.0))

Получается, что и правда, popularity выведен из ранжирования по members на определённый момент времени, но не пересчитывается синхронно с текущим members — отсюда расхождения

### Stats

In [23]:
# stats
detect_anomaly(stats)

,column,role,dtype,null_%,min,max,n_unique,tail,severity,issues,examples
0,mal_id,id,int64,0.00,1,62590.000000,28955,1.000000,ok,ok,
1,watching,count,int64,0.00,0,1838015.000000,5077,35.900000,ok,ok,
2,completed,count,int64,0.00,0,3716436.000000,10080,7.400000,ok,ok,
3,on_hold,count,int64,0.00,0,308583.000000,3764,16.700000,ok,ok,
4,dropped,count,int64,0.00,0,237819.000000,4070,9.600000,ok,ok,
5,plan_to_watch,count,int64,0.00,2,687110.000000,9120,4.400000,ok,ok,
6,total,count,int64,0.00,23,4230824.000000,12265,5.600000,ok,ok,
7,score_1_votes,count,float64,1.49,0,50279.000000,1486,22.400000,ok,ok,
8,score_1_percentage,percentage,float64,1.49,0,100.000000,409,3.700000,ok,ok,
9,score_2_votes,count,float64,1.49,0,36397.000000,1464,16.800000,ok,ok,


Обе pandas-таблицы чистые: жёстких аномалий (`hard_anomalies`) нет, `min`/`max` в разумных границах, а высокий `iqr_outliers_%` у `members`, `scored_by`, `favorites` — это ожидаемый длинный хвост популярности, а не ошибка.

Остаётся `ratings`

### Ratings

Проверим: 
1. username, anime_id (id/ключи) - null%, уникальность
2. status (категориальная) - уникальные значения, их доли
3. score - шкала от 1 до 10
4. num_watched_episodes (счётчик) - распределение

In [ ]:
# username, anime_id
con.sql("""
    SELECT
        SUM(username IS NULL)::DOUBLE / COUNT(*) AS username_null_pct,
        COUNT(DISTINCT username) AS username_cardinality,
        SUM(anime_id IS NULL)::DOUBLE / COUNT(*) AS anime_id_null_pct,
        COUNT(DISTINCT anime_id) AS anime_id_cardinality
    FROM ratings;
""")

┌────────────────────────┬──────────────────────┬───────────────────┬──────────────────────┐
│   username_null_pct    │ username_cardinality │ anime_id_null_pct │ anime_id_cardinality │
│         double         │        int64         │      double       │        int64         │
├────────────────────────┼──────────────────────┼───────────────────┼──────────────────────┤
│ 5.6316110437405056e-08 │               309313 │               0.0 │                29271 │
└────────────────────────┴──────────────────────┴───────────────────┴──────────────────────┘

Тайтлов действительно больше на ~300, похоже, что ratings собирались на балее раннем срезе, часть тайтлов потом удалили из каталога или объединили дубликаты записей. Уникальных пользователей - 309 тыс. 

In [27]:
# status
con.sql("""
    SELECT
        status AS value, COUNT(*) AS cnt
    FROM ratings
    GROUP BY status
    ORDER BY cnt DESC;
""")

┌───────────────┬──────────┐
│     value     │   cnt    │
│    varchar    │  int64   │
├───────────────┼──────────┤
│ completed     │ 79138385 │
│ plan_to_watch │ 31631312 │
│ watching      │  5678167 │
│ dropped       │  4580651 │
│ on_hold       │  3253152 │
│ unknown       │    16690 │
└───────────────┴──────────┘

Ничего неожиданного — 5 ожидаемых статусов + 16690 строк `unknown`, ~0.013% от 124M, это шум. При группировке по статусам её лучше будет исключать явно

In [33]:
# score, num_watched_episodes
con.sql("""
    SELECT
        SUM(score IS NULL)::DOUBLE / COUNT(*) AS score_null_pct,
        MIN(score) AS score_min, MAX(score) AS score_max,
        SUM(score = 0) AS score_zero_cnt,
        MIN(num_watched_episodes) AS ep_min, MAX(num_watched_episodes) AS ep_max,
        SUM(num_watched_episodes < 0) AS ep_negative_cnt,
    FROM ratings;
""")

┌────────────────┬───────────┬───────────┬────────────────┬────────┬────────┬─────────────────┐
│ score_null_pct │ score_min │ score_max │ score_zero_cnt │ ep_min │ ep_max │ ep_negative_cnt │
│     double     │   int64   │   int64   │     int128     │ int64  │ int64  │     int128      │
├────────────────┼───────────┼───────────┼────────────────┼────────┼────────┼─────────────────┤
│            0.0 │         0 │        10 │       54692236 │      0 │  65535 │               0 │
└────────────────┴───────────┴───────────┴────────────────┴────────┴────────┴─────────────────┘

Тут всё хорошо. Посмотрим распределение количества просмотренных эпизодов по статусам

In [35]:
con.sql("""
    SELECT status, 
        AVG(num_watched_episodes) AS avg_watched,
        MIN(num_watched_episodes) AS min_watched,
        MAX(num_watched_episodes) AS max_watched
    FROM ratings
    GROUP BY status;
""")

┌───────────────┬────────────────────┬─────────────┬─────────────┐
│    status     │    avg_watched     │ min_watched │ max_watched │
│    varchar    │       double       │    int64    │    int64    │
├───────────────┼────────────────────┼─────────────┼─────────────┤
│ dropped       │ 11.430921281713013 │           0 │       65535 │
│ plan_to_watch │ 3.4305157180960437 │           0 │       65535 │
│ on_hold       │ 11.640400141155409 │           0 │       65535 │
│ watching      │   26.6173226324622 │           0 │       65535 │
│ completed     │ 15.505988010748514 │           0 │       65535 │
│ unknown       │ 2.8079688436189336 │           0 │        1244 │
└───────────────┴────────────────────┴─────────────┴─────────────┘

Здесь бросаются в глаза 2 вещи: бросают, а также приостанавливают просмотр в среднем на 11-12 серии, а ещё, максимум просмотренных эпизодов во всех 5 статусах - 65535, это ровно битовый предел.

In [36]:
# посмотрим что здесь творится
con.sql("""
    SELECT num_watched_episodes AS эпизодов, count(*) AS строк
    FROM ratings
    WHERE num_watched_episodes > 3000
    GROUP BY 1 ORDER BY 1 DESC LIMIT 5
""").show()

┌──────────┬───────┐
│ эпизодов │ строк │
│  int64   │ int64 │
├──────────┼───────┤
│    65535 │  2691 │
│    65385 │     1 │
│    65322 │     1 │
│    65085 │     1 │
│    65000 │     1 │
└──────────┴───────┘



Значение 65535 встречается 2691 раз, соседние значения (65385, 65322, …) попадаются по одному разу — это не хвост распределения, а одиночный пик на битовом пределе. Учитывая, что самое длинное аниме в базе — 3000 эпизодов, это переполнение или заглушка парсера, а не данные. Для расчёта глубины просмотра такие строки нужно будет отфильтровать: `WHERE num_watched_episodes <= 3000`.

## Дубликаты

В details и stats ключ - `mal_id`, одна строка на тайтл. Проверим, что он действительно уникален и что таблицы связаны один-к-одному: если ключ дублируется, любой `merge` размножит строки и все агрегаты поедут.

In [25]:
check = pd.DataFrame({
    'строк': [len(details), len(stats)],
    'полных дублей': [details.duplicated().sum(), stats.duplicated().sum()],
    'дублей по mal_id': [details['mal_id'].duplicated().sum(), stats['mal_id'].duplicated().sum()],
}, index=['details', 'stats'])
print(check)

# ключи должны совпадать один-в-один в обе стороны
print(f"\nmal_id в details, но не в stats: {(~details['mal_id'].isin(stats['mal_id'])).sum()}")
print(f"mal_id в stats, но не в details: {(~stats['mal_id'].isin(details['mal_id'])).sum()}")

         строк  полных дублей  дублей по mal_id
details  28955              0                 0
stats    28955              0                 0

mal_id в details, но не в stats: 0
mal_id в stats, но не в details: 0


Обе таблицы чистые: 28955 строк, `mal_id` уникален, набор ключей совпадает в обе стороны. Связь строго один-к-одному, `merge` по `mal_id` можно делать спокойно

Теперь ratings. Здесь ключ - `username` + `anime_id`: один пользователь оценивает одно аниме один раз. Проверим дубликаты

In [26]:
con.sql("""
    SELECT count(*) AS всего_строк,
           count(*) - count(DISTINCT (username, anime_id)) AS лишних_строк
    FROM ratings
""").show()

# если дубли есть - посмотрим на них целиком
con.sql("""
    SELECT * FROM ratings
    WHERE (username, anime_id) IN (
        SELECT (username, anime_id) FROM ratings
        GROUP BY username, anime_id HAVING count(*) > 1
    )
    ORDER BY username, anime_id
""").show()

┌─────────────┬──────────────┐
│ всего_строк │ лишних_строк │
│    int64    │    int64     │
├─────────────┼──────────────┤
│   124298357 │            6 │
└─────────────┴──────────────┘

┌────────────┬──────────┬──────────┬───────┬───────────────┬──────────────────────┐
│  username  │ anime_id │  status  │ score │ is_rewatching │ num_watched_episodes │
│  varchar   │  int64   │ varchar  │ int64 │    double     │        int64         │
├────────────┼──────────┼──────────┼───────┼───────────────┼──────────────────────┤
│ Doopelsdoo │    59459 │ watching │     0 │           0.0 │                    8 │
│ Doopelsdoo │    59459 │ watching │     0 │           0.0 │                    9 │
│ Door_mp3   │    59644 │ watching │     8 │           0.0 │                    6 │
│ Door_mp3   │    59644 │ watching │     8 │           0.0 │                    5 │
│ Door_mp3   │    60564 │ watching │     8 │           0.0 │                    5 │
│ Door_mp3   │    60564 │ watching │     8 │           0.

Дубли есть, но их всего 6 пар на 124 млн строк, это ни на что не повлияет 

## Итоги проверки

Данные пригодны для анализа, каких-то критичных проблем нет, всё найденное регулируется фильтрами, восстановление значений не понадобилось..

**Структура и ключи.** `details` и `stats` — по 28955 строк, `mal_id` уникален, наборы ключей совпадают в обе стороны. В `ratings` 124.3 млн строк, 309313 пользователей, 29271 аниме.

**Пропуски.** Пустой `score` в `details` (34.8%) означает «оценки нет»: значений ровно 0 нет ни одного, минимум 1.89, и у всех таких тайтлов пустой `scored_by`. Из них 430 не имеют вообще ни одного голоса — все со статусом `Not yet aired`. MAL не публикует средний балл, если у тайтла пока очень мало голосов. Восстанавливать `score` из распределения голосов не стали — по 2–18 голосам судить о качестве нельзя, а невышедшие тайтлы всё равно вне анализа удовлетворённости (их популярность наследуется от предыдущих сезонов, а нас интересует запуск чего-то нового).

**Убрали из анализа некоторые поля `details`:** выброшены `title_japanese`, `url`, `image_url`, `producers`, `licensors`, `streaming`, `explicit_genres`. `demographics` не используем — 62% пропусков. Вместо `year` (78% пропусков) берём `start_date` (3% пропусков, приведён к datetime). `genres` (21%) и `themes` (41%) годятся, но при разрезах по ним нужно держать в уме размер пропущенной доли.

**Оказалось, что `popularity` и `members` — один и тот же признак.** Ранговая корреляция Спирмена −0.999999, то есть popularity — это просто ранг тайтла по числу добавлений в списки, а не отдельная метрика MAL. Совпадающие значения `popularity` при разных `members` объясняются рассинхроном сбора данных. максимальное расхождение внутри группы — 914 человек при 292799 (0.3%). Не будем использовать members и popularity вместе во избежании мультиколлинеарности.

**Аномалии.** В details и stats "жёстких" аномалий не найдено; высокий процент IQR-выбросов у `members`, `scored_by`, `favorites` — ожидаемо длинный хвост популярности. Одинаковые значения `rank` — не дубли, а один и тот же ранг по равному `score` (хотя логичнее было бы разводить их по `scored_by`). В ratings шкала `score` укладывается в 0–10, отрицательных счётчиков нет, пропусков нет вообще. Реальная аномалия одна — `num_watched_episodes = 65535` (2¹⁶−1) в 2691 строке, соседние значения (65385, 65322, …) встречаются по одному разу, то есть это одиночный пик на битовом пределе, видимо переполнение или заглушка парсера, при расчётах, связанных с глубиной просмотра, нужно будет фильтровать.

**Признак `is_rewatching`.** Не будем его использовать как метрику лояльности. 1.0 стоит лишь у 0.08% строк, а механизма, которым MAL мог бы надёжно фиксировать пересмотры.

### Правила, которые применяем во всех дальнейших расчётах

| Что | Правило | Цена |
|---|---|---|
| Нулевые оценки в `ratings` | `score = 0` — это «не оценил», а не единица. Использовать `NULLIF(score, 0)` либо `WHERE score > 0` | 54.7 млн строк, 44% таблицы: в `plan_to_watch` отсекается 99.3%, в `completed` — 17.2% |
| Глубина просмотра | `WHERE num_watched_episodes <= 3000` (максимум по `details` — 3000 эпизодов) | 2691 строка |
| Статусы | категорию `unknown` исключить явно | 16690 строк, 0.01% |
| Соединение `ratings` × `details` | inner join по `anime_id = mal_id`; 326 аниме в оценках не имеют карточки (вероятно, удалены с MAL после сбора датасета) | 307757 строк, 0.25% |
| Удовлетворённость | считать только по вышедшим тайтлам с непустым `score` | 10073 тайтла вне выборки |
| Популярность | в одном расчёте либо `members`, либо `popularity`, но не оба |  |

Технические фильтры (аномалии, `unknown`, тайтлы без карточки) срезают в сумме около 0.26% строк `ratings` — на метриках охвата это не скажется. Фильтр по оценкам убирает 44% таблицы, но по нему предстоит считать только удовлетворённость, а охват и удержание - по всем строкам.

### Заметки к следующему ноутбуку

- Средняя глубина просмотра у `dropped` — 11.4 эпизода, у `on_hold` — 11.6. Обе близки к концу стандартного 12-серийного сезона, то есть отвал происходит не на первых сериях. Стоит посмотреть распределение, может быть, там два пика — в начале и в конце.
- У `completed` средняя глубина 15.5 против 26.6 у `watching` — ожидаемо, длинные тайтлы дольше висят в процессе.
- 83% досмотревших ставят оценку — отличное значение для расчёта метрики удовлетворённости